# Climate Story — Proposal Visualizations

Static visualizations supporting the proposal. Each plot previews one chapter of the planned scrollytelling narrative and demonstrates that the data supports the story.

**Data source:** NASA NEX-GDDP-CMIP6 via direct HTTPS from AWS S3 (`nex-gddp-cmip6.s3.us-west-2.amazonaws.com`). No authentication required. Citation:

> Thrasher, B., Wang, W., Michaelis, A., Melton, F., Lee, T. and Nemani, R. NASA Global Daily Downscaled Projections, CMIP6. *Scientific Data* 9, 262 (2022).

**Variables available:** `tas` (near-surface air temperature), `tasmax`, `tasmin`, `pr` (precipitation), `huss` (specific humidity), and a few others. **Note:** NEX-GDDP does NOT include `siconc` (sea ice). For the Chapter 2 sea ice visualization, we substitute an Arctic warming amplification map (high-latitude `tas` change). The final project can pull `siconc` from a different source if needed.

**Coverage:** 60°S to 90°N (Antarctica excluded — fine, our narrative focuses on the Arctic and inhabited regions).

**Resolution:** 0.25° daily, downscaled and bias-corrected.

**Scenarios:** SSP2-4.5 (moderate mitigation) and SSP5-8.5 (no mitigation).

**Model:** ACCESS-CM2. One model, one ensemble member (`r1i1p1f1`) — sufficient for proposal statics.

**Plots produced:**
1. Global temperature anomaly choropleth — Ch. 1 *The Warming*
2. Arctic warming amplification — Ch. 2 *The Feedback Loops* (sea-ice substitute)
3. Precipitation anomaly band map — Ch. 3 *The Rainfall Collapse*
4. Compound stress map — Ch. 4 *The Crush*
5. Wet-bulb threshold crossings map — Ch. 4 climax
6. SSP2 vs SSP5 divergence line plot — Ch. 5 *The Fork*

## Setup

Install dependencies. NEX-GDDP files are NetCDF4. `cartopy` for map projections.

In [ ]:
# Run once if needed
# !pip install -q xarray netCDF4 h5netcdf cartopy matplotlib numpy pandas requests tqdm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import requests
from pathlib import Path
from tqdm.auto import tqdm

DATA_DIR = Path("./nex_data")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://nex-gddp-cmip6.s3.us-west-2.amazonaws.com/NEX-GDDP-CMIP6"
MODEL = "ACCESS-CM2"
MEMBER = "r1i1p1f1"
GRID = "gn"

def nex_url(variable, scenario, year):
    """Build the NEX-GDDP-CMIP6 S3 HTTPS URL for a (variable, scenario, year)."""
    fname = f"{variable}_day_{MODEL}_{scenario}_{MEMBER}_{GRID}_{year}.nc"
    return f"{BASE_URL}/{MODEL}/{scenario}/{MEMBER}/{variable}/{fname}"

def download_nex(variable, scenario, year, force=False):
    """Download one year-file to local cache, return the local path."""
    url = nex_url(variable, scenario, year)
    local = DATA_DIR / f"{variable}_{scenario}_{year}.nc"
    if local.exists() and not force:
        return local
    print(f"Downloading {variable} {scenario} {year}...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(local, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))
    return local

def load_nex(variable, scenario, year):
    """Download (if needed) then open as xarray Dataset."""
    path = download_nex(variable, scenario, year)
    return xr.open_dataset(path)

print(f"Setup complete. Data cache: {DATA_DIR.absolute()}")

**Heads up on file sizes.** Each NEX-GDDP yearly file is ~400–750 MB. For proposal statics we'll download strategically: baseline years (2015, 2016) and end-of-century years (2098, 2099), per variable per scenario. Total cache size will be a few GB. If you're on a slow connection, run cells one chapter at a time so you can cancel.

## 1. Global temperature anomaly choropleth — *The Warming*

End-of-century (2098–2099 mean) anomaly relative to a baseline (2015–2016 mean), under SSP5-8.5. Static preview of Chapter 1.

**Note:** Proper climate baselines use 20+ years to average out interannual noise. We're using 2 years here because of file-size constraints for the proposal. The prototype/final will use longer windows.

In [ ]:
tas_2015 = load_nex("tas", "ssp585", 2015)["tas"].mean("time")
tas_2016 = load_nex("tas", "ssp585", 2016)["tas"].mean("time")
tas_2098 = load_nex("tas", "ssp585", 2098)["tas"].mean("time")
tas_2099 = load_nex("tas", "ssp585", 2099)["tas"].mean("time")

baseline = ((tas_2015 + tas_2016) / 2) - 273.15  # K -> C
end_century = ((tas_2098 + tas_2099) / 2) - 273.15
anomaly = end_century - baseline

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.Robinson())
vmax = 10
im = anomaly.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap="RdBu_r", vmin=-vmax, vmax=vmax,
    add_colorbar=False,
)
ax.coastlines(linewidth=0.5)
ax.set_global()
cbar = plt.colorbar(im, orientation="horizontal", pad=0.05, shrink=0.7)
cbar.set_label("Temperature anomaly (°C), 2098-99 vs 2015-16")
plt.title("Ch. 1 — The Warming\nProjected end-of-century temperature change, SSP5-8.5 (ACCESS-CM2)")
plt.savefig("01_warming.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Arctic warming amplification — *The Feedback Loops*

NEX-GDDP doesn't include sea ice concentration (it's a surface-variable-only downscaling), so we substitute the *consequence* of feedback loops: Arctic warming amplification. Polar regions warm 2–4× faster than the global mean precisely *because* of the sea-ice-albedo feedback we want to discuss in Chapter 2.

Polar stereographic projection of the same `tas` anomaly, zoomed to the Arctic. Compare the magnitude here vs the global plot — that gap *is* the feedback signal.

**For the final project**, we can pull `siconc` from a different source (Copernicus Climate Data Store, NSIDC, or raw Pangeo with the auth issue resolved) and restore the original sea ice visual.

In [ ]:
global_mean_warming = float(anomaly.mean().values)
arctic_anomaly = anomaly.sel(lat=slice(60, 90))
arctic_mean_warming = float(arctic_anomaly.mean().values)
amplification = arctic_mean_warming / global_mean_warming

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=ccrs.NorthPolarStereo())
im = anomaly.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap="Reds", vmin=0, vmax=15,
    add_colorbar=False,
)
ax.set_extent([-180, 180, 50, 90], crs=ccrs.PlateCarree())
ax.coastlines(linewidth=0.5)
cbar = plt.colorbar(im, orientation="horizontal", pad=0.05, shrink=0.7)
cbar.set_label("Warming by 2098-99 (°C), SSP5-8.5")
plt.title(
    f"Ch. 2 — The Feedback Loops\n"
    f"Arctic warming amplification: {amplification:.1f}× the global average\n"
    f"(Global mean: {global_mean_warming:.1f}°C, Arctic mean: {arctic_mean_warming:.1f}°C)"
)
plt.savefig("02_arctic_amplification.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Precipitation anomaly band map — *The Rainfall Collapse*

End-of-century precipitation change as a percentage of baseline, brown-blue diverging. Percentage rather than raw mm/day so arid regions don't get drowned out by tropical changes.

In [ ]:
pr_2015 = load_nex("pr", "ssp585", 2015)["pr"].mean("time")
pr_2016 = load_nex("pr", "ssp585", 2016)["pr"].mean("time")
pr_2098 = load_nex("pr", "ssp585", 2098)["pr"].mean("time")
pr_2099 = load_nex("pr", "ssp585", 2099)["pr"].mean("time")

# kg/m^2/s -> mm/day
pr_base = ((pr_2015 + pr_2016) / 2) * 86400
pr_end = ((pr_2098 + pr_2099) / 2) * 86400
pr_pct = ((pr_end - pr_base) / pr_base) * 100
pr_pct = pr_pct.where(pr_base > 0.1)  # mask very arid pixels

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.Robinson())
vmax = 60
im = pr_pct.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap="BrBG", vmin=-vmax, vmax=vmax,
    add_colorbar=False,
)
ax.coastlines(linewidth=0.5)
ax.set_global()
cbar = plt.colorbar(im, orientation="horizontal", pad=0.05, shrink=0.7)
cbar.set_label("Precipitation change (%), 2098-99 vs 2015-16")
plt.title("Ch. 3 — The Rainfall Collapse\nProjected precipitation shift, SSP5-8.5")
plt.savefig("03_precipitation.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Compound stress map — *The Crush*

Counts how many of three climate stresses a grid cell crosses at end-of-century under SSP5-8.5:
1. Warming ≥ 3 °C
2. Precipitation loss ≥ 10%
3. Coastal/low-latitude exposure (proxy for sea level rise vulnerability — placeholder, will be replaced with a proper Low Elevation Coastal Zone mask for the final)

Discrete 0–3 colormap. **New for the final** — not in P3.

In [ ]:
warming_mask = anomaly >= 3.0

# Align precip to the temperature grid (NEX-GDDP variables share a grid, but interp is cheap insurance)
drying_aligned = pr_pct.interp_like(warming_mask).fillna(0)
drying_mask = drying_aligned <= -10.0

# Coastal proxy: low-to-mid latitude (placeholder for LECZ)
lats = anomaly["lat"]
lat_band_1d = (lats > -35) & (lats < 35)
coastal_proxy = lat_band_1d.broadcast_like(anomaly)

stress_count = (
    warming_mask.astype(int)
    + drying_mask.astype(int)
    + coastal_proxy.astype(int)
)

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.Robinson())
cmap = mcolors.ListedColormap(["#f7f7f7", "#fdcc8a", "#fc8d59", "#d7301f"])
im = stress_count.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap=cmap, levels=[-0.5, 0.5, 1.5, 2.5, 3.5],
    add_colorbar=False,
)
ax.coastlines(linewidth=0.5)
ax.set_global()
cbar = plt.colorbar(im, orientation="horizontal", pad=0.05, shrink=0.7, ticks=[0, 1, 2, 3])
cbar.set_label("Number of compounding stresses (warming, drying, coastal exposure)")
plt.title("Ch. 4 — The Crush\nCompound climate stresses by 2098-99, SSP5-8.5")
plt.savefig("04_compound_stress.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Wet-bulb threshold crossings — Ch. 4 climax

Wet-bulb temperature combines heat and humidity into a single "can a human survive outdoors" measure. Sustained wet-bulb ≥ 35 °C exceeds human thermoregulation limits; 31 °C is the conservative danger threshold used in public health literature.

Using Stull's empirical wet-bulb formula from daily max temperature (`tasmax`) and specific humidity (`huss`). Shaded by **number of days per year** with wet-bulb ≥ 31 °C in the year 2099, under SSP5-8.5. **Second new visualization** for the final.

**Heavy cell:** downloads two more ~700 MB files. Skip if pressed for proposal time — the takeaway is documented in the writeup.

In [ ]:
def wet_bulb_stull(T_C, RH_pct):
    """Stull (2011) empirical wet-bulb temperature in °C from T (°C) and RH (%)."""
    RH = RH_pct
    T = T_C
    return (
        T * np.arctan(0.151977 * np.sqrt(RH + 8.313659))
        + np.arctan(T + RH)
        - np.arctan(RH - 1.676331)
        + 0.00391838 * (RH ** 1.5) * np.arctan(0.023101 * RH)
        - 4.686035
    )

def specific_to_relative_humidity(q, T_C, p_pa=101325):
    """Convert specific humidity (kg/kg) to RH (%) given T (°C) and surface pressure (Pa)."""
    es = 610.78 * np.exp((17.27 * T_C) / (T_C + 237.3))  # Tetens, Pa
    e = (q * p_pa) / (0.622 + 0.378 * q)
    return np.clip((e / es) * 100, 0, 100)

# 2099 daily max temp and specific humidity, SSP5-8.5
tmax_ds = load_nex("tasmax", "ssp585", 2099)
huss_ds = load_nex("huss", "ssp585", 2099)

tmax = tmax_ds["tasmax"] - 273.15
huss = huss_ds["huss"]
rh = specific_to_relative_humidity(huss, tmax)
tw = wet_bulb_stull(tmax, rh)

# Days in 2099 with wet-bulb >= 31 °C
danger_days = (tw >= 31).sum("time")

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.Robinson())
im = danger_days.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap="magma_r", vmin=0, vmax=60,
    add_colorbar=False,
)
ax.coastlines(linewidth=0.5)
ax.set_global()
cbar = plt.colorbar(im, orientation="horizontal", pad=0.05, shrink=0.7)
cbar.set_label("Days per year with wet-bulb T ≥ 31 °C (year 2099)")
plt.title("Ch. 4 climax — Wet-bulb danger zone\nAnnual days exceeding human heat-stress threshold, SSP5-8.5")
plt.savefig("05_wet_bulb.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. SSP2 vs SSP5 divergence — *The Fork*

Global mean annual surface temperature anomaly over time, plotted under SSP2-4.5 and SSP5-8.5. This is the literal "fork in the road" — the moral center of the piece.

**Sampling strategy:** Downloading all 85 years for both scenarios would be ~125 GB. Instead, we sample one year per decade. The line plot has fewer points but still shows the divergence cleanly. Final project will use a coarser monthly product or a multi-year average per decade.

In [ ]:
sample_years = [2015, 2025, 2035, 2045, 2055, 2065, 2075, 2085, 2099]

def area_weighted_mean(da):
    """Cosine-latitude weighted global mean."""
    weights = np.cos(np.deg2rad(da["lat"]))
    return da.weighted(weights).mean(["lat", "lon"])

def global_annual_mean_K(scenario, year):
    """Cosine-weighted global mean annual tas for one year, in Kelvin."""
    ds = load_nex("tas", scenario, year)
    annual = ds["tas"].mean("time")
    return float(area_weighted_mean(annual).values)

rows = []
for y in sample_years:
    for scen in ["ssp245", "ssp585"]:
        rows.append({"year": y, "scenario": scen, "tas_K": global_annual_mean_K(scen, y)})
df = pd.DataFrame(rows)

# Use 2015 as the baseline (mean of both scenarios for that year, since they barely differ)
baseline_K = df[df["year"] == 2015]["tas_K"].mean()
df["anomaly_C"] = df["tas_K"] - baseline_K

d245 = df[df["scenario"] == "ssp245"].sort_values("year")
d585 = df[df["scenario"] == "ssp585"].sort_values("year")
gap_2099 = float(d585[d585["year"] == 2099]["anomaly_C"].values[0]
                 - d245[d245["year"] == 2099]["anomaly_C"].values[0])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(d245["year"], d245["anomaly_C"], color="#2c7fb8", linewidth=2.5, marker="o", label="SSP2-4.5 (moderate mitigation)")
ax.plot(d585["year"], d585["anomaly_C"], color="#d7301f", linewidth=2.5, marker="o", label="SSP5-8.5 (no mitigation)")
ax.fill_between(d245["year"], d245["anomaly_C"].values, d585["anomaly_C"].values, color="#d7301f", alpha=0.12)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Year")
ax.set_ylabel("Global mean temperature anomaly (°C)\nvs 2015 baseline")
ax.set_title(f"Ch. 5 — The Fork\nThe gap by 2099: {gap_2099:.1f}°C between futures")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.savefig("06_divergence.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"End-of-century gap between SSP2-4.5 and SSP5-8.5: {gap_2099:.2f} °C")
print("\nFull data:")
print(df.pivot(index='year', columns='scenario', values='anomaly_C').round(2))

## Export for D3 prototype

Bake anomaly grids into JSON files for the browser. One file per (scenario, decade). Coarsened from 0.25° to ~1.5° so file sizes stay under ~500 KB — fast to fetch in GitHub Pages.

Run this after Cell 1 (which produces `anomaly` for SSP5-8.5). Then update the function to loop over decades and scenarios as your bandwidth allows.

In [ ]:
import json
from pathlib import Path

EXPORT_DIR = Path("./prototype_data")
EXPORT_DIR.mkdir(exist_ok=True)

def export_anomaly_grid(da, scenario, decade_label, coarsen_factor=6):
    """Coarsen and serialize an anomaly DataArray to JSON for D3.

    coarsen_factor=6 takes 0.25° -> 1.5°. Drops file size from ~10 MB to ~300 KB.
    """
    coarse = da.coarsen(lat=coarsen_factor, lon=coarsen_factor, boundary="trim").mean()
    out = {
        "scenario": scenario,
        "decade": decade_label,
        "lat": [float(v) for v in coarse["lat"].values],
        "lon": [float(v) for v in coarse["lon"].values],
        # 2D array of anomaly values, row=lat, col=lon. NaN -> null for JSON.
        "values": [
            [None if np.isnan(v) else round(float(v), 2) for v in row]
            for row in coarse.values
        ],
    }
    path = EXPORT_DIR / f"temp_anomaly_{scenario}_{decade_label}.json"
    with open(path, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print(f"Wrote {path} ({path.stat().st_size // 1024} KB)")

def anomaly_for(scenario, start_year, end_year, baseline_da):
    """Compute anomaly for a (scenario, year-window) vs a baseline DataArray (already in °C)."""
    yrs = [load_nex("tas", scenario, y)["tas"].mean("time") for y in range(start_year, end_year + 1)]
    decade_mean = (sum(yrs) / len(yrs)) - 273.15
    return decade_mean - baseline_da

# Re-use the baseline from Plot 1 (already in °C)
baseline_C = baseline  # from Plot 1 cell

# Pick decades. Each downloads 2 year-files per scenario per decade if not cached.
# For the prototype, 3 decades x 2 scenarios = 12 files = ~6-9 GB of downloads.
# Cut the decade list down if bandwidth-constrained.
decades = [
    ("2030s", 2035, 2036),
    ("2060s", 2065, 2066),
    ("2090s", 2098, 2099),
]

for scenario in ["ssp245", "ssp585"]:
    for label, ys, ye in decades:
        a = anomaly_for(scenario, ys, ye, baseline_C)
        export_anomaly_grid(a, scenario, label)

print("\nDone. Copy ./prototype_data/*.json into your prototype repo's data/ folder.")

## Notes for the proposal writeup

**Data source:** NASA NEX-GDDP-CMIP6 (downscaled, bias-corrected CMIP6 projections at 0.25° resolution), ACCESS-CM2 model, accessed via direct HTTPS from the AWS Public Dataset registry.

**Reused from P3 (with feedback applied):** 1, 3.
- #1 uses a symmetric clipped scale so the Arctic doesn't blow out the rest of the map.
- #3 switches from raw mm/day to percentage change with arid masking.

**Substituted:** 2.
- NEX-GDDP doesn't include sea ice. Substituted with Arctic warming amplification, which is *caused by* the same albedo feedback. For the final, we can pull `siconc` from a separate source (Copernicus, NSIDC, or Pangeo with auth resolved) and restore the original sea ice visual.

**New for the final:** 4, 5, 6.
- Compound stress, wet-bulb threshold crossings, and the SSP2 vs SSP5 head-to-head divergence are all net-new vs P3.

**Known limitations to call out at the feedback meeting:**
- Single model (ACCESS-CM2), single ensemble member. Multi-model ensemble means would be more rigorous for the final.
- 2-year baselines and end-of-century averages instead of 20-year windows. Done because of file-size constraints (each year-file is ~500–750 MB).
- The compound-stress map uses a latitude-band proxy for coastal exposure. A proper Low Elevation Coastal Zone (LECZ) mask would be more accurate — but it's an external dataset, so check whether that's in scope.
- Antarctica is excluded from NEX-GDDP coverage (60°S southern limit). Fine for this story since the narrative focuses on the Arctic and inhabited regions.